In [8]:
# make sure you have Github copilot installed, search it in the VSCode extension marketplace, it will make your coding much easier
import sys 
sys.path.append('/host/c/Users/ROG/Documents/Github')
import os
import torch
import numpy as np
import nibabel as nb
import CTDenoising_Diffusion_N2N.noise2noise.model as noise2noise
import CTDenoising_Diffusion_N2N.functions_collection as ff
import CTDenoising_Diffusion_N2N.Build_lists.Build_list as Build_list
import CTDenoising_Diffusion_N2N.noise2noise.Generator as Generator

# if it says no module named ... (e.g., lpips), do the following:
# 1. go to powershell, type wsl, enter wsl
# 2. in wsl, type: sudo docker container ls, you will see the container id of your running container
# 3. type: sudo docker container exec -it -u 0 <container_id> bash
# 4. now you are inside the container root, type: pip install lpips

### Step 1: define pre-trained model

In [9]:
trial_name = 'noise2noise'
epoch = 78
# define your own saved model path and prediction save path
trained_model_filename = os.path.join('/host/d/file/', trial_name, 'model-' + str(epoch)+ '.pt')
save_folder = os.path.join('/host/d/file/pre', trial_name, 'pred_images'); os.makedirs(save_folder, exist_ok=True)

In [10]:
### parameters no need to change
image_size = [512,512]

histogram_equalization = True
background_cutoff = -1000
maximum_cutoff = 2000
normalize_factor = 'equation' 

### step 2: build patient list - testing, batch 5

In [11]:
# example excel path (you can find it in this repo - example data folder)
# build_sheet =  Build_list.Build(os.path.join('/host/d/Github/CTDenoising_Diffusion_N2N/example_data/patient_lists/patient_list_supervised.xlsx'))
# use the spreadsheet I gave to you and put the path here:
build_sheet =  Build_list.Build(os.path.join('/host/d/file/fixedCT_static_simulation_train_test_gaussian_local.xlsx'))

_,patient_id_list,patient_subid_list,random_num_list, condition_list, x0_list = build_sheet.__build__(batch_list = [5])

### step 3: build model

In [12]:
# build model
model = noise2noise.Unet2D(
    init_dim = 16,
    channels = 2, 
    out_dim = 1,
    dim_mults = (2,4,8,16),
    full_attn = (None,None, False, True),
    act = 'ReLU',
)

in out is :  [(16, 32), (32, 64), (64, 128), (128, 256)]


### step 4: main

In [13]:
# load histogram equalization pre-saved files
# change to your own path, they are in this repo - example data folder
bins = np.load('/host/d/file/histogram_equalization/bins.npy') 
bins_mapped = np.load('/host/d/file/histogram_equalization/bins_mapped.npy')

In [ ]:
# main
for i in range(0, 1):#patient_id_list.shape[0]):
    patient_id = patient_id_list[i]
    patient_subid = patient_subid_list[i]
    random_num = random_num_list[i]
    x0_file = x0_list[i]
    condition_file = condition_list[i]

    print(i,patient_id, patient_subid, random_num)

    # # get the ground truth image
    # gt_img = nb.load(x0_file)
    # affine = gt_img.affine; gt_img = gt_img.get_fdata()[:,:,30:80]

    # get the condition image
    condition_img = nb.load(condition_file).get_fdata()[:,:,30:80]
    affine = nb.load(condition_file).affine
    

    # make folders
    ff.make_folder([os.path.join(save_folder, patient_id), os.path.join(save_folder, patient_id, patient_subid), os.path.join(save_folder, patient_id, patient_subid, 'random_' + str(random_num))])
    save_folder_case = os.path.join(save_folder, patient_id, patient_subid, 'random_' + str(random_num), 'epoch' + str(epoch)); os.makedirs(save_folder_case, exist_ok=True)

    # save condition image
    nb.save(nb.Nifti1Image(condition_img, affine), os.path.join(save_folder_case,'condition_img.nii.gz'))

    # # generator
    generator = Generator.Dataset_2D(
        img_list = np.array([condition_file]),
        image_size = image_size,

        num_slices_per_image = 50,
        random_pick_slice = False,
        slice_range = [30,80],

        bins = bins,
        bins_mapped = bins_mapped,
        histogram_equalization = histogram_equalization,
        background_cutoff = background_cutoff,
        maximum_cutoff = maximum_cutoff,
        normalize_factor = normalize_factor,)

    # ====== 小函数：用现有 sampler 导出 Stage-I 缓存（不改库）======
    def export_stage1_cache(sampler, trained_model_filename, save_dir):
        os.makedirs(save_dir, exist_ok=True)
        sampler.load_model(trained_model_filename)
        device = sampler.device
        sampler.ema.ema_model.eval()

        def _spatial_mean_2d(x):
            # N2N 输入是 2D：(B, C, H, W)
            return x.mean(dim=(2, 3), keepdim=True)

        idx = 0
        with torch.inference_mode():
            for batch in sampler.dl:
                # 兼容 (input, gt) 或只有 input 的数据集
                if isinstance(batch, (list, tuple)):
                    x = batch[0]      # noisy 输入
                else:
                    x = batch

                x = x.to(device)                      # (B, 2, H, W)
                y_bar = sampler.ema.ema_model(x)      # Φ(x) = ȳ，形状 (B, 1, H, W)
                eps   = x[:, :1] - y_bar              # 用中间通道作为 y 的对齐对象就会混乱，这里取 x 的中心视图可选
                # 注意：如果你希望严格按论文构造 eps = noisy_center - ȳ，确保上面这行用的是与 ȳ 对应的 noisy 切片

                mu    = _spatial_mean_2d(eps)         # (B, 1, 1, 1)
                eps0  = eps - mu                      # 零均值残差
                y_bar0 = y_bar + mu                   # 保持 x_center = ȳ0 + ε̄0

                torch.save({
                    'y_bar':  y_bar0.detach().cpu().float(),
                    'eps_bar': eps0.detach().cpu().float()
                }, os.path.join(save_dir, f'stage1_{idx:06d}.pt'))
                idx += 1

    # ====== 调用它 ======
    sampler = noise2noise.Sampler(model, generator, batch_size=1, image_size=image_size)
    stage1_dir = os.path.join(save_folder_case, 'stage1_cache')
    os.makedirs(stage1_dir, exist_ok=True)
    export_stage1_cache(sampler, trained_model_filename, stage1_dir)
    print(f'[OK] Stage-I cache exported to: {stage1_dir}')
    # ===============================================


    


0 00214841 0000455418 0


NameError: name 'pred_img_final' is not defined